# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

### Set the default catalog and read in the dataframes

In [0]:
%sql
USE jarvis_training_catalog.pgexercises;

In [0]:
bookings_df = spark.sql("select * from bookings")
facilities_df = spark.sql("select * from facilities")
members_df = spark.sql("select * from members")

## Join

### Q1

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
'''
SELECT cdb.starttime 
FROM cd.members AS cdm 
JOIN
cd.bookings AS cdb
ON cdm.memid = cdb.memid
WHERE cdm.firstname = 'David' AND
cdm.surname = 'Farrell';
'''

j1 = bookings_df.join(members_df, "memid", 'inner').filter((f"firstname = 'David' AND surname = 'Farrell'")).select("starttime")
display(j1)

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


### Q2

How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

https://pgexercises.com/questions/joins/simplejoin2.html

In [0]:
'''
SELECT cdb.starttime AS start, cdf.name AS name
FROM cd.bookings AS cdb JOIN
cd.facilities AS cdf
ON cdb.facid = cdf.facid
WHERE DATE(cdb.starttime) = '2012-09-21' AND
cdf.name LIKE 'Tennis Court%'
ORDER BY start ASC;
'''
from pyspark.sql.functions import to_date

j2 = bookings_df.join(facilities_df, "facid", 'inner').filter((facilities_df.name.like('Tennis Court%')) & (to_date(bookings_df.starttime) == '2012-09-21')).select("starttime", "name").orderBy("starttime")
display(j2)

starttime,name
2012-09-21T08:00:00.000Z,Tennis Court 2
2012-09-21T08:00:00.000Z,Tennis Court 1
2012-09-21T09:30:00.000Z,Tennis Court 1
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T11:30:00.000Z,Tennis Court 2
2012-09-21T12:00:00.000Z,Tennis Court 1
2012-09-21T13:30:00.000Z,Tennis Court 1
2012-09-21T14:00:00.000Z,Tennis Court 2
2012-09-21T15:30:00.000Z,Tennis Court 1
2012-09-21T16:00:00.000Z,Tennis Court 2


### Q3
How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

https://pgexercises.com/questions/joins/self2.html

In [0]:
'''
SELECT cdm1.firstname AS memfname,
cdm1.surname AS memsname,
cdm2.firstname AS recfname,
cdm2.surname AS recsname
FROM cd.members AS cdm1 LEFT JOIN
cd.members AS cdm2
ON cdm1.recommendedby = cdm2.memid
ORDER BY memsname, memfname ASC;
'''
from pyspark.sql.functions import col

j3 = members_df.alias('m1').join(
    members_df.alias('m2'),
    col('m1.recommendedby') == col('m2.memid'), 'left').select(
    col('m1.firstname').alias('memfname'),
    col('m1.surname').alias('memsname'),
    col('m2.firstname').alias('recfname'),
    col('m2.surname').alias('recsname')
).orderBy('memsname', 'memfname')
display(j3)

memfname,memsname,recfname,recsname
Florence,Bader,Ponder,Stibbons
Anne,Baker,Ponder,Stibbons
Timothy,Baker,Jemima,Farrell
Tim,Boothe,Tim,Rownam
Gerald,Butters,Darren,Smith
Joan,Coplin,Timothy,Baker
Erica,Crumpet,Tracy,Smith
Nancy,Dare,Janice,Joplette
David,Farrell,null,null
Jemima,Farrell,null,null


### Q4
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.

https://pgexercises.com/questions/joins/threejoin.html

In [0]:
'''
SELECT DISTINCT
CONCAT(cdm.firstname, ' ', cdm.surname) AS member,
cdf.name AS facility
FROM cd.members cdm JOIN cd.bookings cdb
ON cdm.memid = cdb.memid JOIN cd.facilities cdf
ON cdb.facid = cdf.facid
WHERE cdf.name LIKE 'Tennis Court%'
ORDER BY member, facility ASC
'''
from pyspark.sql.functions import concat, lit

j4 = bookings_df.join(members_df, 'memid', 'inner').join(facilities_df, 'facid', 'inner').filter(facilities_df.name.like('Tennis Court%')).select(
    concat(members_df.firstname, lit(' '), members_df.surname).alias('member'),
    facilities_df.name.alias('facility')).dropDuplicates().orderBy('member', 'facility')

display(j4)

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


### Q5
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

https://pgexercises.com/questions/joins/sub.html

In [0]:
'''
SELECT DISTINCT CONCAT(cdm1.firstname, ' ', cdm1.surname) AS member,
(SELECT CONCAT(cdm2.firstname, ' ', cdm2.surname) 
 FROM cd.members AS cdm2
WHERE cdm1.recommendedby = cdm2.memid)
 FROM cd.members AS cdm1
 ORDER BY member ASC;
'''

j5 = members_df.alias('m1').join(members_df.alias('m2'), col('m1.recommendedby') == col('m2.memid'), 'left').select(
    concat(col('m1.firstname'), lit(' '), col('m1.surname')).alias('member'),
    concat(col('m2.firstname'), lit(' '), col('m2.surname')).alias('recommendedby')
).dropDuplicates().orderBy('member')
display(j5)

member,recommendedby
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null
David Farrell,null
David Jones,Janice Joplette
David Pinker,Jemima Farrell
Douglas Jones,David Jones
Erica Crumpet,Tracy Smith


## Aggregation


### Q1
Produce a count of the number of recommendations each member has made. Order by member ID.

https://pgexercises.com/questions/aggregates/count3.html

In [0]:
'''
SELECT recommendedby, COUNT(*) 
FROM cd.members
WHERE recommendedby > 0
GROUP BY recommendedby
ORDER BY recommendedby ASC;
'''

a1 = members_df.filter(col('recommendedby') > 0).groupBy("recommendedby").count().orderBy("recommendedby")
display(a1)

recommendedby,count
1,5
2,3
3,1
4,2
5,1
6,1
9,2
11,1
13,2
15,1


### Q2
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.


https://pgexercises.com/questions/aggregates/fachours.html

In [0]:
'''
SELECT cdf.facid, SUM(cdb.slots) AS "Total Slots"
FROM cd.bookings cdb JOIN
cd.facilities cdf
ON cdb.facid = cdf.facid
GROUP BY cdf.facid
ORDER BY cdf.facid;
'''
# use named aggregation with aliases
from pyspark.sql.functions import sum, col
a2 = bookings_df.join(facilities_df, "facid", 'inner').groupBy('facid').agg(sum('slots').alias('Total Slots')).orderBy("facid")

display(a2)

facid,Total Slots
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


### Q3
Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

https://pgexercises.com/questions/aggregates/fachoursbymonth.html

In [0]:
'''
SELECT cdf.facid, SUM(cdb.slots) AS "Total Slots"
FROM cd.facilities AS cdf JOIN
cd.bookings AS cdb
ON cdb.facid = cdf.facid
WHERE DATE(cdb.starttime) BETWEEN '2012-09-01' AND '2012-09-30'
GROUP BY cdf.facid
ORDER BY 2 ASC;
'''

a3 = bookings_df.join(facilities_df, 'facid', 'inner').filter(to_date(bookings_df.starttime).between('2012-09-01', '2012-09-30')).groupBy('facid').agg(sum('slots').alias('Total Slots')).orderBy("Total Slots")

display(a3)

facid,Total Slots
5,122
3,422
7,426
8,471
6,540
2,570
1,588
0,591
4,648


### Q4
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

https://pgexercises.com/questions/aggregates/fachoursbymonth2.html

In [0]:
'''
SELECT cdf.facid, EXTRACT(MONTH FROM cdb.starttime), SUM(cdb.slots) AS "Total Slots"
FROM
cd.bookings AS cdb JOIN
cd.facilities AS cdf
ON cdb.facid = cdf.facid
WHERE EXTRACT(YEAR FROM cdb.starttime) = 2012
GROUP BY cdf.facid, 2 
ORDER BY cdf.facid, 2 ASC;
'''

from pyspark.sql.functions import month, year

a4 = bookings_df.join(facilities_df, 'facid', 'inner').filter(year(bookings_df.starttime) == 2012).groupBy("facid", month(bookings_df.starttime)).agg(
    sum('slots').alias('Total Slots')).orderBy("facid", month(bookings_df.starttime))

display(a4)

facid,month(starttime),Total Slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483
1,9,588
2,7,180
2,8,459
2,9,570
3,7,104


### Q5
Find the total number of members (including guests) who have made at least one booking.

https://pgexercises.com/questions/aggregates/members1.html

In [0]:
'''
SELECT COUNT(memid) FROM
(SELECT memid, COUNT(*) 
FROM cd.bookings
GROUP BY memid)
WHERE count > 1
'''
from pyspark.sql.functions import count

a5_subquery = bookings_df.groupBy('memid').sum('slots').filter(col('sum(slots)') > 1)
a5 = a5_subquery.agg(count(col('memid')))

display(a5)

count(memid)
30


### Q6
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.

https://pgexercises.com/questions/aggregates/nbooking.html

In [0]:
'''

'''
from pyspark.sql.functions import min

a6 = members_df.join(bookings_df, "memid", 'inner').filter(bookings_df.starttime > '2012-09-01').groupBy(
    members_df.surname, members_df.firstname, members_df.memid).agg(min(bookings_df.starttime)).orderBy(members_df.memid)

display(a6)

surname,firstname,memid,min(starttime)
GUEST,GUEST,0,2012-09-01T08:00:00.000Z
Smith,Darren,1,2012-09-01T09:00:00.000Z
Smith,Tracy,2,2012-09-01T11:30:00.000Z
Rownam,Tim,3,2012-09-01T16:00:00.000Z
Joplette,Janice,4,2012-09-01T15:00:00.000Z
Butters,Gerald,5,2012-09-02T12:30:00.000Z
Tracy,Burton,6,2012-09-01T15:00:00.000Z
Dare,Nancy,7,2012-09-01T12:30:00.000Z
Boothe,Tim,8,2012-09-01T08:30:00.000Z
Stibbons,Ponder,9,2012-09-01T11:00:00.000Z


## String

### Q1
Output the names of all members, formatted as 'Surname, Firstname'

https://pgexercises.com/questions/string/concat.html

In [0]:
'''
SELECT CONCAT(surname, ', ' , firstname)
FROM cd.members;
'''

s1 = members_df.select(concat(members_df.firstname, lit(' '), members_df.surname))

display(s1)

"concat(firstname, , surname)"
GUEST GUEST
Darren Smith
Tracy Smith
Tim Rownam
Janice Joplette
Gerald Butters
Burton Tracy
Nancy Dare
Tim Boothe
Ponder Stibbons


### Q2
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.

https://pgexercises.com/questions/string/case.html

In [0]:
'''
SELECT * FROM cd.members WHERE name LIKE 'Tennis%
'''

s2 = facilities_df.filter(facilities_df.name.like('Tennis%')).select('*')

display(s2)

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5,25,10000,200
1,Tennis Court 2,5,25,8000,200


### Q3
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

https://pgexercises.com/questions/string/reg.html

In [0]:
'''
SELECT memid, telephone
FROM cd.members
WHERE telephone LIKE '(___)%'
ORDER BY memid ASC;
'''

s3 = members_df.filter(members_df.telephone.like('(___)%'))[['memid', 'telephone']].orderBy('memid')

display(s3)

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


### Q4
You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

https://pgexercises.com/questions/string/substr.html

In [0]:
from pyspark.sql.functions import substr, count

'''
SELECT SUBSTR(surname, 1, 1) AS letter, COUNT(*)
FROM cd.members
GROUP BY letter
ORDER BY letter ASC;
'''

s4 = members_df.groupBy(members_df.surname.substr(1, 1).alias('letter')).agg(count('*')).orderBy('letter')

display(s4)

letter,count(1)
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


### Q5
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.

https://pgexercises.com/questions/date/series.html

In [0]:
from pyspark.sql.functions import expr, explode, sequence

'''
SELECT GENERATE_SERIES('2012-10-01'::date, '2012-10-31'::date, '1 day'::interval);
'''

s5 = spark.createDataFrame([{'date': 1}]).select(
    explode(
        sequence(
            to_date(lit('2012-10-01')),       
            to_date(lit('2012-10-31')),       
            expr("INTERVAL 1 DAY")            
        )
    ).alias('calendar_date'))

display(s5)

calendar_date
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


### Q6
Return a count of bookings for each month, sorted by month

https://pgexercises.com/questions/date/bookingspermonth.html

In [0]:
from pyspark.sql.functions import date_trunc

'''
SELECT DATE_TRUNC('month', starttime) AS month,
COUNT(*)
FROM cd.bookings
GROUP BY month
ORDER BY month; 
'''

s6 = bookings_df.groupBy(
    date_trunc('month', col('starttime')).alias('month')).agg(count('*').alias('total_bookings')).orderBy('month')

display(s6) 

month,total_bookings
2012-07-01T00:00:00.000Z,658
2012-08-01T00:00:00.000Z,1472
2012-09-01T00:00:00.000Z,1913
2013-01-01T00:00:00.000Z,1
